In [2]:
from cfg import (
    iit_g_base_path,
    iit_g_labels_path,
    iit_g_images_path
)
import os 

# clean up aar - filter out images with no labels 
labels = os.listdir(iit_g_labels_path)
labels_with_graphs = []

for path in labels: 
    with open(os.path.join(iit_g_labels_path, path), 'r') as f: 
        lines = f.readlines()
        if len(lines) > 0: 
            labels_with_graphs.append(path)
        else: 
            print(f"no labels found for {path}")


no labels found for ar_daimler_2018_noneng_54_jpg.rf.77f0103f3572c60599ee565cadc026d8.txt
no labels found for ar_atoss_2018_noneng_39_jpg.rf.370da4aab1d910641875f5a81df0996d.txt
no labels found for ar_siemens_2004_noneng_1_jpg.rf.4ab8b80c9e9d17ed8beafb24023daae5.txt
no labels found for ar_enel_2017_eng_300_jpg.rf.12a95eecf1a8621628cae5644a696f03.txt
no labels found for ar_ge_2016_eng_43_jpg.rf.b41ec08ace1f0ed37ffe819f6fd7b25e.txt
no labels found for ar_bayer_2027_noneng_40_jpg.rf.793fbc27ee414f102b9efc13250bbc25.txt
no labels found for ar_deutschebank_2008_noneng_58_jpg.rf.c39999e6d5d59408c626707446accdab.txt
no labels found for ar_fca_2013_eng_32_jpg.rf.0648ff4d699f00c22c7984e80a6fd772.txt
no labels found for ar_alstom_2016_eng_52_jpg.rf.71365c63ab3f42c595031880fdd44d7c.txt
no labels found for ar_deutschebank_2007_eng_1_jpg.rf.ff5f477589ee6246a4fa55871f831b17.txt
no labels found for ar_basf_2015_noneng_147_jpg.rf.7e6a746a03bd8bc4873162238ef1c4af.txt
no labels found for ar_deutschebank

In [3]:
print(f"""
    Done cleaning up IIT dataset.
    Images with labels: {len(labels_with_graphs)}
    Images without labels: {len(labels) - len(labels_with_graphs)}
      """)


    Done cleaning up IIT dataset.
    Images with labels: 1352
    Images without labels: 7894
      


IIT dataset cleaned, now to copy the data to separate directories

In [7]:
for path in labels_with_graphs: 
    # replace txt with jpg
    img_path = path.replace('txt', 'jpg')
    os.system(f"cp {os.path.join(iit_g_images_path, img_path)} {os.path.join(iit_g_base_path, 'images_cleaned', img_path)}")
    os.system(f"cp {os.path.join(iit_g_labels_path, path)} {os.path.join(iit_g_base_path, 'labels_cleaned', path)}")

Sanity check to make sure that processed dataset can be mapped back to original dataset

In [31]:
from cfg import (
    iit_base_path,
    iit_train_path,
    iit_val_path,
    iit_test_path
)

# let's combine the train, val, and test for now
# we can split them later
iit_combined_images = os.listdir(
    iit_train_path) + os.listdir(iit_val_path) + os.listdir(iit_test_path)
iit_combined_images = [os.path.splitext(path)[0] for path in iit_combined_images]
print(f"Combined images: {len(iit_combined_images)}")

# sanity check to check if our processed images are in the combined list containing the original images
for path in labels_with_graphs: 
    processed_name = path[:path.find('_jpg')]
    if processed_name not in iit_combined_images: 
        print(f"processed image {processed_name} not found in combined images")

Combined images: 13408


# Train test split

In [25]:
iit_train_processed_lst = os.listdir(iit_train_path)
iit_train_processed_lst = [os.path.splitext(path)[0] for path in iit_train_processed_lst]

iit_val_processed_lst = os.listdir(iit_val_path)
iit_val_processed_lst = [os.path.splitext(path)[0] for path in iit_val_processed_lst]

iit_test_processed_list = os.listdir(iit_test_path)
iit_test_processed_list = [os.path.splitext(path)[0] for path in iit_test_processed_list]

iit_g_train, iit_g_val, iit_g_test = [], [], []

for path in labels_with_graphs: 
    img_path = path.replace('txt', 'jpg')
    processed_name = img_path[:img_path.find('_jpg')]

    # if in train, val, or test, add to respective list
    if processed_name in iit_train_processed_lst: 
        iit_g_train.append(img_path)
    elif processed_name in iit_val_processed_lst: 
        iit_g_val.append(img_path)
    elif processed_name in iit_test_processed_list: 
        iit_g_test.append(img_path)
    else: 
        print(f"processed image {processed_name} not found in train, val, or test")


In [30]:
print(len(iit_g_train), len(iit_g_val), len(iit_g_test))
print(f"ratio of train to val to test: {round(len(iit_g_train) / len(labels_with_graphs),3)} : {round(len(iit_g_val) / len(labels_with_graphs),3)} : {round(len(iit_g_test) / len(labels_with_graphs),3)}")


1035 61 256
ratio of train to val to test: 0.766 : 0.045 : 0.189


In [32]:
# save out the train, val, and test lists
with open(os.path.join(iit_g_base_path, 'iit_train.json'), 'w') as f: 
    f.write(str(iit_g_train))

with open(os.path.join(iit_g_base_path, 'iit_val.json'), 'w') as f:
    f.write(str(iit_g_val))

with open(os.path.join(iit_g_base_path, 'iit_test.json'), 'w') as f:
    f.write(str(iit_g_test))
